In [1]:
import numpy as np
import pandas as pd

input_file = "train.csv"
df_train = pd.read_csv(input_file)
df_train.shape

(7352, 563)

In [2]:
df_train["subject"].value_counts()

subject
25    409
21    408
26    392
30    383
28    382
27    376
23    372
17    368
16    366
19    360
1     347
29    344
3     341
15    328
6     325
14    323
22    321
11    316
7     308
5     302
8     281
Name: count, dtype: int64

In [3]:
df_train["Activity"].value_counts()

Activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64

In [4]:
from sklearn.preprocessing import OrdinalEncoder

df_train_cat = df_train[["Activity"]]
ordinal_encoder = OrdinalEncoder()
y_train = ordinal_encoder.fit_transform(df_train_cat).flatten()
ordinal_encoder.categories_

[array(['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS',
        'WALKING_UPSTAIRS'], dtype=object)]

In [5]:
# 특성 엔지니어링 (누수 방지): 교차검증의 각 fold 안에서, 그 fold 에 들어온 행만으로 계산한다
#  같은 사람의 앞뒤 측정 구간 평균 (짧은 창 9, 긴 창 51) 을 원본 특성에 덧붙인다
#  정답(Activity)은 사용하지 않고, 학습 fold 와 시험 fold 의 값이 서로 섞이지 않는다 (subject 는 묶음 기준으로만 쓰고 특성에서는 뺌)
def har_features(X):
    subject = X["subject"]
    num = X.drop(columns=["subject"])
    parts = [num.to_numpy()]
    for w in [9, 51]:
        roll = num.copy()
        for s in subject.unique():
            idx = (subject == s).to_numpy()
            roll.loc[idx] = num.loc[idx].rolling(w, center=True, min_periods=1).mean()
        parts.append(roll.to_numpy())
    return np.hstack(parts)

X_all = df_train.drop(["Activity"], axis=1)          # 561 특성 + subject
X_raw = df_train.drop(["subject", "Activity"], axis=1).to_numpy()
har_features(X_all.iloc[:500]).shape

(500, 1683)

In [6]:
# 1) 모델 비교: 원본 특성 vs 내 특성, 여러 분류 모델 (10-fold, 섞지 않음)
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "SGD": SGDClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=3000),
    "LinearSVC": LinearSVC(max_iter=10000),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "SVC": SVC(C=10),
}
for name, model in models.items():
    raw = cross_val_score(model, X_raw, y_train, cv=10, scoring="accuracy").mean()
    fe = cross_val_score(make_pipeline(FunctionTransformer(har_features), model), X_all, y_train, cv=10, scoring="accuracy").mean()
    print(f"{name:20s} 원본 {raw:.4f}   내 특성 {fe:.4f}")

SGD                  원본 0.9436   내 특성 0.9546


LogisticRegression   원본 0.9480   내 특성 0.9645


LinearSVC            원본 0.9512   내 특성 0.9680


DecisionTree         원본 0.8558   내 특성 0.8715


RandomForest         원본 0.9308   내 특성 0.9366


SVC                  원본 0.9539   내 특성 0.9672


In [7]:
# 2) 하이퍼파라미터 탐색: 가장 좋은 SVC 의 C, gamma (Grid Search, 5-fold)
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(FunctionTransformer(har_features), SVC())
param_grid = {"svc__C": [10, 100, 1000], "svc__gamma": ["scale", 0.0005, 0.002]}
grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_all, y_train)
print(grid_search.best_params_, grid_search.best_score_)
for mean, params in zip(grid_search.cv_results_["mean_test_score"], grid_search.cv_results_["params"]):
    print(f"{mean:.4f}", params)

{'svc__C': 100, 'svc__gamma': 'scale'} 0.9676336612143158
0.9657 {'svc__C': 10, 'svc__gamma': 'scale'}
0.9606 {'svc__C': 10, 'svc__gamma': 0.0005}
0.9653 {'svc__C': 10, 'svc__gamma': 0.002}
0.9676 {'svc__C': 100, 'svc__gamma': 'scale'}
0.9655 {'svc__C': 100, 'svc__gamma': 0.0005}
0.9671 {'svc__C': 100, 'svc__gamma': 0.002}
0.9676 {'svc__C': 1000, 'svc__gamma': 'scale'}
0.9655 {'svc__C': 1000, 'svc__gamma': 0.0005}
0.9671 {'svc__C': 1000, 'svc__gamma': 0.002}


In [8]:
# 3) 최종 모델 10-fold 교차 검증 (섞지 않음)
final_clf = grid_search.best_estimator_

cv_res = cross_val_score(final_clf, X_all, y_train, cv=10, scoring="accuracy")
cv_res

array([0.99456522, 0.94021739, 0.9755102 , 0.87482993, 0.96598639,
       0.96054422, 0.99047619, 0.98503401, 1.        , 0.99047619])

In [9]:
np.mean(cv_res)

np.float64(0.9677639751552796)

In [10]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

y_pred = cross_val_predict(final_clf, X_all, y_train, cv=10)
confusion_matrix(y_train, y_pred)

array([[1407,    0,    0,    0,    0,    0],
       [  13, 1237,   36,    0,    0,    0],
       [   0,   49, 1325,    0,    0,    0],
       [   0,    0,    0, 1154,   13,   59],
       [   0,    0,    0,   19,  960,    7],
       [   0,    0,    0,   16,   25, 1032]])